# Module 15 — Systems Level Structures SkipLists BloomFilters LRU

## What you will discover

Every cell below runs this module's **real** problem-bank solutions and asserts
their behaviour. Nothing here prints a claim it has not computed.

The assertions are lifted directly from `problems/tests/`, so they cannot drift
from the implementations — if a signature changes, the tests break first.

**One cell near the end is deliberately broken.** Fixing it is the exercise.

## Setup

The solutions directory goes on `sys.path` relative to this notebook's own
location. Never hard-code an absolute path — `tools/check_links.py` fails the
build on them, because a path with a username in it works on exactly one
machine.

In [ ]:
import sys
import time
from pathlib import Path

import pytest   # some assertions check that an invalid input RAISES
sys.path.insert(0, str(Path.cwd() / "problems" / "solutions"))

from p01_lru_cache import simulate_lru
from p02_bloom_filter import bloom_check
from p03_skip_list import simulate_skip_list

print("module 15: Systems Level Structures SkipLists BloomFilters LRU")
print("problems available:", 8)
for name in ['p01_lru_cache', 'p02_bloom_filter', 'p03_skip_list', 'p04_ring_buffer', 'p05_lfu_cache', 'p06_hyperloglog', 'p07_count_min_sketch', 'p08_consistent_hash_ring']:
    print(f"  {name}")

## 1. Baseline — `p01_lru_cache`

The first property, asserted rather than printed. Read the assertions before
running: each one names a specific input class, and most cross-check against an
independent brute force over the same data.

In [ ]:
ops = [("put", 1, 1), ("put", 2, 2), ("get", 1, 0), ("put", 3, 3), ("get", 2, 0)]
assert simulate_lru(2, ops) == [1, -1]
# A miss on an empty cache.
assert simulate_lru(1, [("get", 1, 0)]) == [-1]
# Updating a key must not evict anything.
ops = [("put", 1, 1), ("put", 1, 2), ("get", 1, 0)]
assert simulate_lru(1, ops) == [2]
# A get counts as a use, protecting the key from eviction.
ops = [("put", 1, 1), ("put", 2, 2), ("get", 1, 0), ("put", 3, 3),
       ("get", 1, 0), ("get", 2, 0), ("get", 3, 0)]
assert simulate_lru(2, ops) == [1, 1, -1, 3]
# A put also counts as a use.
ops = [("put", 1, 1), ("put", 2, 2), ("put", 1, 10), ("put", 3, 3),
       ("get", 2, 0), ("get", 1, 0)]
assert simulate_lru(2, ops) == [-1, 10]
# Capacity 1 evicts on every new key.
ops = [("put", 1, 1), ("put", 2, 2), ("get", 1, 0), ("get", 2, 0)]
assert simulate_lru(1, ops) == [-1, 2]
with pytest.raises(ValueError):
    simulate_lru(0, [("get", 1, 0)])
# Cross-check against a naive list-based model.
import random
random.seed(21)
cap = 4
order: list[int] = []
model: dict[int, int] = {}
seq, expected = [], []
for _ in range(2000):
    k = random.randint(1, 8)
    if random.random() < 0.5:
        v = random.randint(0, 100)
        seq.append(('put', k, v))
        if k in model:
            order.remove(k)
        model[k] = v
        order.append(k)
        if len(order) > cap:
            del model[order.pop(0)]
    else:
        seq.append(('get', k, 0))
        if k in model:
            order.remove(k)
            order.append(k)
            expected.append(model[k])
        else:
            expected.append(-1)
assert simulate_lru(cap, seq) == expected

print("all assertions held")

## 2. Predict before you run

A Bloom filter holds 800 items in 8192 bits using 3 hashes. Predict the false-positive rate when membership requires ALL three bits, and when it requires ANY one. Then predict the false-*negative* rate in each case.

Commit to an answer before executing the next cell. Predicting and being wrong
is what makes the correction stick; reading the output first does not.

In [ ]:
# The core guarantee: no false negatives, ever.
words = ["cat", "dog", "bird", "fish", "horse", "mouse"]
verdicts = bloom_check(words, words)
assert all(verdicts), "an inserted item must never report absent"
# Empty filter: nothing can be present.
assert bloom_check([], ["anything"]) == [False]
assert bloom_check([], []) == []
# Deterministic across calls.
a = bloom_check(["x", "y"], ["x", "z", "y", "w"])
b = bloom_check(["x", "y"], ["x", "z", "y", "w"])
assert a == b, "the filter must be reproducible"
# With a generous bit budget, false positives should be rare.
inserted = [f"item-{i}" for i in range(200)]
absent = [f"missing-{i}" for i in range(2000)]
fp = sum(bloom_check(inserted, absent, bits=65536, hashes=4))
assert fp < 100, f"false positive rate looks too high: {fp}/2000"
# And the guarantee still holds at that scale.
assert all(bloom_check(inserted, inserted, bits=65536, hashes=4))
# A tiny filter produces many false positives - which is allowed,, # and is exactly the trade being made.
assert all(bloom_check(inserted, inserted, bits=64, hashes=3))
with pytest.raises(ValueError):
    bloom_check(["a"], ["a"], bits=0)

print("all assertions held")

## 3. Measurement

Claims about complexity are claims about wall-clock behaviour at scale, so they
have to be measured rather than asserted from the shape of the code.

In [ ]:
started = time.perf_counter()

ops = [("insert", 3), ("insert", 1), ("search", 3), ("items", 0),
       ("delete", 3), ("search", 3)]
assert simulate_skip_list(ops) == [True, [1, 3], True, False]
# Searching an empty structure.
assert simulate_skip_list([("search", 1), ("items", 0)]) == [False, []]
# Inserts are idempotent.
ops = [("insert", 5), ("insert", 5), ("items", 0)]
assert simulate_skip_list(ops) == [[5]]
# Deleting something absent returns False.
assert simulate_skip_list([("delete", 9)]) == [False]
# Contents stay sorted regardless of insertion order.
ops = [("insert", v) for v in (5, 1, 9, 3, 7)] + [("items", 0)]
assert simulate_skip_list(ops) == [[1, 3, 5, 7, 9]]
# Negative values.
ops = [("insert", -3), ("insert", 0), ("insert", -7), ("items", 0)]
assert simulate_skip_list(ops) == [[-7, -3, 0]]
with pytest.raises(ValueError):
    simulate_skip_list([("frobnicate", 1)])
# Cross-check against a plain sorted set over a long sequence.
import random as _r
_r.seed(31)
model: set[int] = set()
seq, expected = [], []
for _ in range(1500):
    v = _r.randint(1, 60)
    r = _r.random()
    if r < 0.45:
        seq.append(('insert', v))
        model.add(v)
    elif r < 0.7:
        seq.append(('search', v))
        expected.append(v in model)
    elif r < 0.9:
        seq.append(('delete', v))
        expected.append(v in model)
        model.discard(v)
    else:
        seq.append(('items', 0))
        expected.append(sorted(model))
assert simulate_skip_list(seq) == expected

elapsed = (time.perf_counter() - started) * 1000
print(f"all assertions held in {elapsed:.2f} ms")

## 4. Fix this cell

The values below are **wrong on purpose**. Run it, read the failure, work out
the right numbers from the cells above, and correct them in place.

Change only the expected values — not the code that computes them.

In [ ]:
import os

# DELIBERATELY BROKEN - two expected values, both wrong. Fix in place.

expected_problem_count = 99      # how many problems does this module ship?
expected_solution_count = 99     # how many reference solutions are on disk?

problem_files = sorted(
    f for f in os.listdir(Path.cwd() / "problems") if f.startswith("p") and f.endswith(".py")
)
solution_files = sorted(
    f for f in os.listdir(Path.cwd() / "problems" / "solutions")
    if f.startswith("p") and f.endswith(".py")
)

assert expected_problem_count == len(problem_files), (
    f"expected {expected_problem_count} problems, found {len(problem_files)}"
)
assert expected_solution_count == len(solution_files), (
    f"expected {expected_solution_count} solutions, found {len(solution_files)}"
)
print("Both match. Every problem has exactly one reference solution.")

## Takeaways

1. A Bloom filter's guarantee is one-sided, and `any` destroys its usefulness without breaking it.
2. An LRU needs a hash map for lookup and a linked list for ordering; neither alone works.
3. A performance-shaped bug with no functional symptom is the hardest kind to find.

### Where to go next

- [`01_README.md`](01_README.md) — the concepts in depth
- [`problems/README.md`](problems/README.md) — all 8 problems, with hint ladders
- [`debug_lab/SYMPTOMS.md`](debug_lab/SYMPTOMS.md) — planted defects that exit 0
- [Pattern Recognition Guide](../PATTERN_RECOGNITION_GUIDE.md) — attacking an unseen problem